Step 0 - Explanation

In [1]:
"""
Preprocessing Notebook: FASTA -> TXT -> Masked TXT
---------------------------------------------------
Reads Salmonella_1.fna to Salmonella_11.fna from the current directory.

Step 1 - FASTA to TXT:
- Salmonella_1 to Salmonella_10 --> combined into train.txt (10 genomes)
- Salmonella_11                  --> test.txt

Step 2 - Masking (replicates exact transformer internal masking):
- Pattern: i % 5 == (i // 5) % 5
- train_m.txt  : M only
- train_mr.txt : M + R (85% masked, 15% replaced with wrong nucleotide)
- Same for test files
- Seed 42 for reproducibility

Output: 6 files total
- train.txt, train_m.txt, train_mr.txt
- test.txt,  test_m.txt,  test_mr.txt
"""

'\nPreprocessing Notebook: FASTA -> TXT -> Masked TXT\n---------------------------------------------------\nReads Salmonella_1.fna to Salmonella_11.fna from the current directory.\n\nStep 1 - FASTA to TXT:\n- Salmonella_1 to Salmonella_10 --> combined into train.txt (10 genomes)\n- Salmonella_11                  --> test.txt\n\nStep 2 - Masking (replicates exact transformer internal masking):\n- Pattern: i % 5 == (i // 5) % 5\n- train_m.txt  : M only\n- train_mr.txt : M + R (85% masked, 15% replaced with wrong nucleotide)\n- Same for test files\n- Seed 42 for reproducibility\n\nOutput: 6 files total\n- train.txt, train_m.txt, train_mr.txt\n- test.txt,  test_m.txt,  test_mr.txt\n'

Step 1 - Libraries

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import random
from collections import Counter


# Seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print(f"Seed set to {SEED}")

Seed set to 42


Step 2 - Main code

In [3]:
def parse_fasta(filepath: str) -> str:
    sequence_parts = []
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                continue
            sequence_parts.append(line.upper())
    return "".join(sequence_parts)


def main():
    class args:
        input_dir  = "."
        output_dir = "."

    os.makedirs(args.output_dir, exist_ok=True)

    train_files = [f"Salmonella_{i}.fna" for i in range(1, 11)]  # 1-10
    test_file   = "Salmonella_11.fna"                             # 11

    train_output      = os.path.join(args.output_dir, "train.txt")
    total_train_bases = 0

    with open(train_output, "w") as out_f:
        for fname in train_files:
            fpath = os.path.join(args.input_dir, fname)
            if not os.path.exists(fpath):
                raise FileNotFoundError(f"Expected file not found: {fpath}")
            seq = parse_fasta(fpath)
            out_f.write(seq + "")
            total_train_bases += len(seq)
            print(f"  [train] {fname}: {len(seq):,} bases")

    print(f"Training file written  --> {train_output}")
    print(f"  Total training bases   : {total_train_bases:,} (10 genomes)")

    test_output = os.path.join(args.output_dir, "test.txt")
    fpath = os.path.join(args.input_dir, test_file)
    if not os.path.exists(fpath):
        raise FileNotFoundError(f"Expected file not found: {fpath}")
    seq = parse_fasta(fpath)

    with open(test_output, "w") as out_f:
        out_f.write(seq + "")

    print(f"  [test]  {test_file}: {len(seq):,} bases")
    print(f"  Test file written      --> {test_output}")
    print(f"Done! train.txt and test.txt ready — proceeding to masking.")

main()

  [train] Salmonella_1.fna: 5,203,495 bases
  [train] Salmonella_2.fna: 4,952,765 bases
  [train] Salmonella_3.fna: 4,638,880 bases
  [train] Salmonella_4.fna: 4,951,383 bases
  [train] Salmonella_5.fna: 4,831,756 bases
  [train] Salmonella_6.fna: 4,857,466 bases
  [train] Salmonella_7.fna: 5,255,665 bases
  [train] Salmonella_8.fna: 5,027,605 bases
  [train] Salmonella_9.fna: 4,983,099 bases
  [train] Salmonella_10.fna: 4,913,592 bases
Training file written  --> .\train.txt
  Total training bases   : 49,615,706 (10 genomes)
  [test]  Salmonella_11.fna: 4,932,877 bases
  Test file written      --> .\test.txt
Done! train.txt and test.txt ready — proceeding to masking.


In [4]:

def count_nucleotides(filepath: str):
    """Count A, T, G, C (and any other symbols) across all sequences in a txt file."""
    counts = Counter()
    total = 0
    with open(filepath, "r") as f:
        for line in f:
            seq = line.strip().upper()
            counts.update(seq)
            total += len(seq)
    return counts, total


counts, total = count_nucleotides("train.txt")

print(f"Nucleotide composition of training data ({total:,} bases total)")
print("-" * 50)
for base in ["A", "T", "G", "C"]:
    n = counts.get(base, 0)
    print(f"  {base} : {n:>14,}  ({n/total*100:5.2f}%)")

# Any non-ATGC symbols (e.g. N)
other = {b: c for b, c in counts.items() if b not in "ATGC"}
if other:
    print("\n  Other symbols:")
    for b, c in sorted(other.items()):
        print(f"  {b} : {c:>14,}  ({c/total*100:5.2f}%)")

# GC content — a standard genomics statistic worth reporting
gc = counts.get("G", 0) + counts.get("C", 0)
print(f"\n  GC content: {gc/total*100:.2f}%")

Nucleotide composition of training data (49,615,706 bases total)
--------------------------------------------------
  A :     11,881,112  (23.95%)
  T :     11,881,557  (23.95%)
  G :     12,933,215  (26.07%)
  C :     12,919,822  (26.04%)

  GC content: 52.11%


In [5]:
NUCLEOTIDES = ['A', 'T', 'G', 'C']

def mask_only_sequence(seq: str, start_window: int = 0) -> str:
    """M only masking. Position cycle backwards: 5,4,3,2,1"""
    seq = list(seq)
    num_windows = len(seq) // 5
    for i in range(num_windows):
        window_idx    = start_window + i
        pos_in_window = window_idx % 5
        mask_pos      = i * 5 + pos_in_window
        if mask_pos >= len(seq):
            continue
        seq[mask_pos] = 'M'
    return "".join(seq)


def mixed_mask_sequence(seq: str, start_window: int = 0) -> str:
    seq = list(seq)
    num_windows = len(seq) // 5

    for i in range(num_windows):
        window_idx    = start_window + i
        pos_in_window = window_idx % 5
        mask_pos      = i * 5 + pos_in_window

        if mask_pos >= len(seq):
            continue

        # 85% chance mask with M, 15% chance replace with wrong nucleotide
        if random.random() < 0.85:
            seq[mask_pos] = 'M'
        else:
            original      = seq[mask_pos]
            choices       = [n for n in NUCLEOTIDES if n != original]
            seq[mask_pos] = random.choice(choices)

    return "".join(seq)


def mask_file(input_path: str, output_m_path: str, output_mr_path: str):
    """Read txt file, produce both _m and _mr versions."""
    window_counter = 0
    
    # Cleaned up the explicit '\n' text characters from the line continuations
    with open(input_path, "r") as in_f, \
         open(output_m_path,  "w") as m_f, \
         open(output_mr_path, "w") as mr_f:
         
        for line in in_f:
            seq = line.strip()
            if not seq:
                continue

            masked_m  = mask_only_sequence (seq, start_window=window_counter)
            masked_mr = mixed_mask_sequence(seq, start_window=window_counter)

            # Added actual newlines to the write statements so your output 
            # files don't end up as one massive, unreadable line.
            m_f.write(masked_m  + "\n")
            mr_f.write(masked_mr + "\n")

            total          = len(seq)
            m_count        = masked_m.count('M')
            mr_m_count     = masked_mr.count('M')
            mr_r_count     = sum(1 for o, m in zip(seq, masked_mr) if m != o and m != 'M')
            print(f"  Bases: {total:,} | M only: {m_count:,} ({m_count/total*100:.1f}%) | MR masked: {mr_m_count:,} replaced: {mr_r_count:,} ({(mr_m_count+mr_r_count)/total*100:.1f}%)")

            window_counter += total // 5


# Reset seed before masking for reproducibility
random.seed(SEED)
np.random.seed(SEED)

print("Masking training set...")
mask_file("train.txt", "train_m.txt", "train_mr.txt")
print("  train_m.txt and train_mr.txt written")

print("Masking test set...")
mask_file("test.txt", "test_m.txt", "test_mr.txt")
print("  test_m.txt and test_mr.txt written")

print("Done! All 11 files ready:")
print("  train.txt, train_m.txt, train_mr.txt")
print("  test.txt,  test_m.txt,  test_mr.txt")

Masking training set...
  Bases: 49,615,706 | M only: 9,923,141 (20.0%) | MR masked: 8,435,114 replaced: 1,488,027 (20.0%)
  train_m.txt and train_mr.txt written
Masking test set...
  Bases: 4,932,877 | M only: 986,575 (20.0%) | MR masked: 838,175 replaced: 148,400 (20.0%)
  test_m.txt and test_mr.txt written
Done! All 11 files ready:
  train.txt, train_m.txt, train_mr.txt
  test.txt,  test_m.txt,  test_mr.txt
